# 📊 JobLens — NLP Exploration Notebook

Use this to:
- Test the pipeline interactively before running the API
- Generate charts for your GitHub README
- Understand how each model behaves on real data

**Run all cells top to bottom.**

In [ ]:
# Install dependencies (only needed once)
# !pip install transformers torch pandas matplotlib seaborn wordcloud scikit-learn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Dark theme to match the app
plt.rcParams.update({
    'figure.facecolor':  '#0d1117',
    'axes.facecolor':    '#161b22',
    'axes.edgecolor':    '#21262d',
    'text.color':        '#e6edf3',
    'axes.labelcolor':   '#8b949e',
    'xtick.color':       '#8b949e',
    'ytick.color':       '#8b949e',
    'grid.color':        '#21262d',
    'axes.grid':         True,
})

print('✅ Libraries loaded')

In [ ]:
# Sample dataset — replace with real Glassdoor/Indeed reviews
raw_reviews = [
    ("Great work life balance and amazing team culture. Management is very supportive.", "TechCo"),
    ("Pay is below market rate and hard to get promoted. Management doesn't listen to feedback.", "TechCo"),
    ("Good benefits package but the workload is unsustainably intense. Expect long hours.", "TechCo"),
    ("Fantastic learning opportunities. The company really invests in your career growth.", "StartupX"),
    ("Toxic work environment with extremely high turnover. Would not recommend to anyone.", "StartupX"),
    ("Decent salary and solid work life balance. Nothing exciting but very stable company.", "StartupX"),
    ("Best place I have ever worked. Collaborative culture and excellent leadership team.", "MegaCorp"),
    ("Underpaid for the amount of work demanded. Benefits are okay but not exceptional.", "MegaCorp"),
    ("Remote work options are great. Management trusts employees to get work done independently.", "MegaCorp"),
    ("Poor work life balance and micromanagement is a serious problem at every level.", "TechCo"),
]

df = pd.DataFrame(raw_reviews, columns=['review', 'company'])
print(f'Dataset: {len(df)} reviews across {df.company.nunique()} companies')
df.head()

In [ ]:
# ── SENTIMENT ANALYSIS ──────────────────────────────────────
from transformers import pipeline

print('Loading sentiment model... (downloads ~500MB on first run)')
sentiment_pipe = pipeline(
    'sentiment-analysis',
    model='cardiffnlp/twitter-roberta-base-sentiment-latest',
    top_k=None,
    truncation=True,
    max_length=512,
)
print('✅ Model ready')

def run_sentiment(text):
    results = sentiment_pipe(text)[0]
    scores  = {r['label'].lower(): r['score'] for r in results}
    pos = scores.get('positive', 0.0)
    neg = scores.get('negative', 0.0)
    neu = scores.get('neutral',  0.0)
    label = max([('positive', pos), ('negative', neg), ('neutral', neu)], key=lambda x: x[1])[0]
    return label, round(pos, 3), round(neg, 3), round(neu, 3)

df[['sentiment', 'pos', 'neg', 'neu']] = df['review'].apply(
    lambda x: pd.Series(run_sentiment(x))
)

df[['review', 'sentiment', 'pos', 'neg', 'neu']]

In [ ]:
# ── PLOT 1: Sentiment Distribution ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Sentiment Analysis Results', color='#e6edf3', fontsize=14, fontweight='bold')

# Pie
counts = df['sentiment'].value_counts()
colors = {'positive': '#7ee787', 'negative': '#f78166', 'neutral': '#8b949e'}
axes[0].pie(
    counts.values,
    labels=counts.index,
    autopct='%1.0f%%',
    colors=[colors.get(l, '#484f58') for l in counts.index],
    startangle=90,
    textprops={'color': '#e6edf3', 'fontsize': 12},
)
axes[0].set_title('Overall Sentiment Split', color='#e6edf3')

# Bar
x = range(len(df))
axes[1].bar(x, df['pos'], color='#7ee787', alpha=0.8, label='Positive')
axes[1].bar(x, df['neg'], color='#f78166', alpha=0.8, label='Negative', bottom=df['pos'])
axes[1].set_xlabel('Review #')
axes[1].set_ylabel('Score')
axes[1].set_title('Sentiment Scores per Review', color='#e6edf3')
axes[1].legend()
axes[1].set_xticks(x)
axes[1].set_xticklabels([str(i+1) for i in x])

plt.tight_layout()
plt.savefig('../data/sentiment_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: data/sentiment_chart.png  ← screenshot this for your README!')

In [ ]:
# ── ZERO-SHOT TOPIC CLASSIFICATION ──────────────────────────
print('Loading zero-shot model... (downloads ~1.6GB on first run)')
zero_shot = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')
print('✅ Model ready')

TOPICS = [
    'company culture and work environment',
    'salary and compensation',
    'management and leadership',
    'career growth and promotions',
    'work life balance and flexibility',
    'employee benefits and perks',
]

SHORT = {
    'company culture and work environment': 'culture',
    'salary and compensation':              'salary',
    'management and leadership':            'management',
    'career growth and promotions':         'growth',
    'work life balance and flexibility':    'work_life_balance',
    'employee benefits and perks':          'benefits',
}

def get_top_topic(text):
    result = zero_shot(text[:512], TOPICS, multi_label=False)
    return SHORT.get(result['labels'][0], result['labels'][0]), round(result['scores'][0], 3)

df[['top_topic', 'topic_score']] = df['review'].apply(
    lambda x: pd.Series(get_top_topic(x))
)

df[['review', 'sentiment', 'top_topic', 'topic_score']]

In [ ]:
# ── PLOT 2: Topic Distribution ───────────────────────────────
topic_counts = df['top_topic'].value_counts()

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(
    topic_counts.index,
    topic_counts.values,
    color='#00e5ff',
    alpha=0.75,
    edgecolor='#21262d',
)
ax.set_xlabel('Number of Reviews')
ax.set_title('Most Discussed Topics Across Reviews', color='#e6edf3', fontsize=13, fontweight='bold')
for bar, val in zip(bars, topic_counts.values):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            str(val), va='center', color='#e6edf3', fontsize=11)

plt.tight_layout()
plt.savefig('../data/topic_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: data/topic_chart.png  ← screenshot this for your README!')

In [ ]:
# ── PLOT 3: Sentiment by Company ────────────────────────────
company_sentiment = df.groupby(['company', 'sentiment']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4))
company_sentiment.plot(
    kind='bar',
    ax=ax,
    color={'positive': '#7ee787', 'negative': '#f78166', 'neutral': '#8b949e'},
    edgecolor='#21262d',
    alpha=0.85,
)
ax.set_xlabel('Company')
ax.set_ylabel('Review Count')
ax.set_title('Sentiment Breakdown by Company', color='#e6edf3', fontsize=13, fontweight='bold')
ax.legend(title='Sentiment')
plt.xticks(rotation=0)

plt.tight_layout()
plt.savefig('../data/company_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: data/company_sentiment.png')

In [ ]:
# ── WORD CLOUD ───────────────────────────────────────────────
try:
    from wordcloud import WordCloud
    all_text = ' '.join(df['review'])
    wc = WordCloud(
        width=900, height=400,
        background_color='#0d1117',
        colormap='cool',
        max_words=80,
        contour_width=0,
    ).generate(all_text)
    plt.figure(figsize=(12, 5), facecolor='#0d1117')
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.tight_layout(pad=0)
    plt.savefig('../data/wordcloud.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    print('✅ Saved: data/wordcloud.png  ← great for README header!')
except ImportError:
    print('Install wordcloud first: pip install wordcloud')

In [ ]:
# ── SUMMARY TABLE ────────────────────────────────────────────
print('\n📊 FULL RESULTS TABLE')
print('=' * 80)
for _, row in df.iterrows():
    print(f"Review : {row['review'][:65]}...")
    print(f"Company: {row['company']}")
    print(f"Label  : {row['sentiment'].upper()}  |  Topic: {row['top_topic']}  |  Confidence: {row['topic_score']}")
    print('-' * 80)

print('\n✅ Notebook complete. Use the charts in your GitHub README!')
print('   data/sentiment_chart.png')
print('   data/topic_chart.png')
print('   data/company_sentiment.png')
print('   data/wordcloud.png')